In [1]:
import os, findspark, pyspark, random
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType
# os.environ['PYSPARK_PYTHON'] = '/usr/bin/python'
# os.environ['PYSPARK_DRIVER_PYTHON'] = '/usr/bin/python'
findspark.init(os.getenv('SPARK_HOME'))
#  .set('spark.master', 'local[*]')
#  .set('spark.master', 'yarn') \
#  .setAppName('JupyterLab')
spark_conf = SparkConf() \
  .set('spark.app.name', 'JupyterLab') \
  .set('spark.driver.memory', '3g') \
  .set('spark.executor.memory', '1g') \
  .set('spark.submit.deployMode', 'client') \
  .set("spark.scheduler.mode", "FAIR") \
  .set("spark.shuffle.compress", "true") \
  .set("spark.speculation", "true") \
  .set("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
  .set("spark.sql.sources.partitionOverwriteMode", "dynamic") \
  .set("spark.hadoop.parquet.enable.summary-metadata", "false") \
  .set("spark.sql.parquet.mergeSchema", "false") \
  .set("spark.sql.parquet.filterPushdown", "true") \
  .set("spark.sql.hive.metastorePartitionPruning", "true")

In [2]:
spark = SparkSession \
  .builder \
  .config(conf=spark_conf) \
  .enableHiveSupport() \
  .getOrCreate()
sc = spark.sparkContext
spark

## Simple Dataframe

In [3]:
df = spark.createDataFrame([1, 2, 3, 4], IntegerType())
df.show()

+-----+
|value|
+-----+
|    1|
|    2|
|    3|
|    4|
+-----+



In [4]:
df2 = spark.createDataFrame(
  [('2018-08-20', 1.0), ('2018-08-21', 2.0), ('2018-08-24', 3.0)], 
  ['time', 'v']
)
df2.show()

+----------+---+
|      time|  v|
+----------+---+
|2018-08-20|1.0|
|2018-08-21|2.0|
|2018-08-24|3.0|
+----------+---+



## Compute PI

In [5]:
num_samples = 100000000
def inside(p):
  x, y = random.random(), random.random()
  return x*x + y*y < 1
count = sc.parallelize(range(0, num_samples)).filter(inside).count()
pi = 4 * count / num_samples
print(pi)

3.14025616


In [7]:
spark.stop()